# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
from importlib.metadata import version

for package in ("requests", "beautifulsoup4", "schedule", "pytest"):
    print(f"{package}: {version(package)}")

requests: 2.34.2
beautifulsoup4: 4.15.0
schedule: 1.2.2
pytest: 9.2.0


In [2]:
import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

project_root = Path.cwd()
if not (project_root / "scraper.py").is_file():
    project_root = project_root.parent
if not (project_root / "scraper.py").is_file():
    raise FileNotFoundError("Откройте ноутбук из папки проекта.")
os.chdir(project_root)
print("Библиотеки загружены, рабочая папка проекта установлена.")

Библиотеки загружены, рабочая папка проекта установлена.


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
def get_book_data(
    book_url: str, session: requests.Session | None = None
) -> dict:
    """Получить данные о книге со страницы Books to Scrape.

    Args:
        book_url (str): Полный URL страницы книги.
        session (requests.Session | None): Общая HTTP-сессия или None
            для отдельного запроса.

    Returns:
        dict: Название (title, str), цена в фунтах (price, float),
            рейтинг от 1 до 5 (rating, int), остаток (stock, int),
            описание (description, str) и все характеристики
            Product Information (product_info, dict).
    """
    client = session if session is not None else requests
    response = client.get(book_url, timeout=30)
    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")
    main = soup.select_one(".product_main")
    if main is None:
        raise ValueError(f"Не найдены данные книги: {book_url}")

    title = main.find("h1").get_text(strip=True)
    price_text = main.select_one(".price_color").get_text(strip=True)
    price = float(price_text.replace("£", ""))

    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    rating_classes = main.select_one(".star-rating")["class"]
    rating = next(
        rating_map[name]
        for name in rating_classes
        if name in rating_map
    )

    availability = main.select_one(".availability").get_text(
        " ", strip=True
    )
    stock_match = re.search(r"\d+", availability)
    stock = int(stock_match.group()) if stock_match else 0

    description_tag = soup.select_one("#product_description + p")
    description = (
        description_tag.get_text(" ", strip=True)
        if description_tag is not None
        else ""
    )

    product_info = {}
    for row in soup.select("table.table-striped tr"):
        key = row.find("th").get_text(strip=True)
        value = row.find("td").get_text(" ", strip=True)
        product_info[key] = value

    return {
        "title": title,
        "price": price,
        "rating": rating,
        "stock": stock,
        "description": description,
        "product_info": product_info,
    }

In [4]:
book_url = (
    "https://books.toscrape.com/catalogue/"
    "a-light-in-the-attic_1000/index.html"
)
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 3,
 'stock': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def scrape_books(is_save: bool = False) -> list[dict]:
    """Собрать данные обо всех книгах каталога Books to Scrape.

    Args:
        is_save (bool): Если True, сохранить результат в books_data.txt
            в текущей рабочей папке. По умолчанию False.

    Returns:
        list[dict]: Список словарей с данными всех книг каталога.
    """
    books = []
    page_url = "https://books.toscrape.com/catalogue/page-1.html"
    retries = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )

    with requests.Session() as session:
        session.mount("https://", HTTPAdapter(max_retries=retries))
        while True:
            response = session.get(page_url, timeout=30)
            response.raise_for_status()
            response.encoding = "utf-8"
            soup = BeautifulSoup(response.text, "html.parser")
            book_links = soup.select("article.product_pod h3 a")

            if not book_links:
                raise ValueError(f"Нет ссылок на книги: {page_url}")

            for link in book_links:
                book_url = urljoin(response.url, link["href"])
                books.append(get_book_data(book_url, session=session))
                time.sleep(0.2)

            print(f"Собрано книг: {len(books)}", flush=True)
            next_page = soup.select_one("li.next a")
            if next_page is None:
                break
            page_url = urljoin(response.url, next_page["href"])

    if is_save:
        with open("books_data.txt", "w", encoding="utf-8") as file:
            json.dump(books, file, ensure_ascii=False, indent=4)

    return books

In [6]:
res = scrape_books(is_save=True)
print(type(res), len(res))
print("Уникальных книг:", len({b["product_info"]["UPC"] for b in res}))

with open("books_data.txt", encoding="utf-8") as file:
    saved_books = json.load(file)
assert saved_books == res
Path("artifacts").mkdir(exist_ok=True)
shutil.copy2("books_data.txt", "artifacts/books_data.txt")
print("Книг в сохранённом файле:", len(saved_books))

Собрано книг: 20


Собрано книг: 40


Собрано книг: 60


Собрано книг: 80


Собрано книг: 100


Собрано книг: 120


... (41 outputs left out)

Собрано книг: 960


Собрано книг: 980


Собрано книг: 1000


<class 'list'> 1000
Уникальных книг: 1000
Книг в сохранённом файле: 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
def run_schedule(run_time: str = "19:00", once: bool = False) -> None:
    """Запускать сбор с сохранением файла ежедневно в указанное время.

    Args:
        run_time (str): Локальное время запуска в формате ЧЧ:ММ.
        once (bool): Если True, завершиться после первой выгрузки.
            Используется для проверки расписания в ноутбуке.

    Returns:
        None: Результаты сохраняются в books_data.txt.
    """
    scheduler = schedule.Scheduler()
    job = scheduler.every().day.at(run_time).do(
        scrape_books, is_save=True
    )
    print(f"Следующий запуск: {job.next_run}", flush=True)

    try:
        while True:
            scheduler.run_pending()
            if once and job.last_run is not None:
                print("Выгрузка по расписанию завершена.", flush=True)
                break
            time.sleep(1)
    except KeyboardInterrupt:
        print("Планировщик остановлен.")

print("Ежедневный запуск: 19:00 по локальному времени компьютера.")

Ежедневный запуск: 19:00 по локальному времени компьютера.


### Проверка запуска по расписанию

Проверка назначает полный сбор на ближайшую следующую минуту в формате ЧЧ:ММ и завершается после одной успешной выгрузки. Для постоянной работы ежедневно в 19:00 используется `run_schedule()` или `python scraper.py --schedule`.

In [8]:
test_time = (datetime.now() + timedelta(minutes=1)).strftime("%H:%M")
print("Время проверки:", test_time)
run_schedule(run_time=test_time, once=True)

with open("books_data.txt", encoding="utf-8") as file:
    scheduled_books = json.load(file)
assert len(scheduled_books) == len(res)
shutil.copy2("books_data.txt", "artifacts/books_data.txt")
print("Книг после запуска по расписанию:", len(scheduled_books))

Время проверки: 21:05
Следующий запуск: 2026-11-15 21:05:00


Собрано книг: 20


Собрано книг: 40


Собрано книг: 60


Собрано книг: 80


Собрано книг: 100


... (43 outputs left out)

Собрано книг: 980


Собрано книг: 1000


Выгрузка по расписанию завершена.


Книг после запуска по расписанию: 1000


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [9]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_scraper.py",
     "-v", "--color=no"],
    cwd=project_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Тесты завершились с ошибкой."

============================= test session starts ==============================
platform linux -- Python 3.12.14, pytest-9.2.0, pluggy-1.7.0 -- /workspace/scratch/18326fd8f176/.task_env/bin/python
cachedir: .pytest_cache
rootdir: /workspace/scratch/18326fd8f176/clean_project
configfile: pytest.ini
plugins: platformdirs-4.14.5
collecting ... collected 3 items

tests/test_scraper.py::test_get_book_data PASSED                         [ 33%]
tests/test_scraper.py::test_scrape_books_without_saving PASSED           [ 66%]
tests/test_scraper.py::test_scrape_books_with_saving PASSED              [100%]

============================== 3 passed in 0.31s ===============================



## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: ...`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: ...`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: ...`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: ...`

**5. Самооценка по критериям задания**:

`Ответ: ...`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

